# Steam Game Genre Analysis
## Fase 2: Data Insameling en Verwerking

**Onderwerp:** “Watter genre werk?” — watter genre is die gewildste, watter kenmerke dryf sukses, en hoe lyk die kommentaar binne elke genre?

**Datastel:** 33 speletjies oor 15 genres, ~163 000 Engelse resensies (voor skoonmaak), die nuutste ~1 jaar per speletjie (mediaan 365 dae)

**Databronne:** Steam API — resensies, speletjiebesonderhede, huidige spelertellings

**Tegnologieë:** Python, pandas, numpy, requests, matplotlib

**Fase-omvang:** Slegs Fase 2 (insameling + verwerking). Fase 3 (EDA) word aan die einde slegs beplan.

**Herhaalbaarheid:** Verwyder `data/raw/*.csv` en `data/processed/*.csv` en hardloop die notebook weer.

**Naatlike verandering na professor se terugvoer:** Scrape na Engels-only; parameters empiries geregverdig.

---
## 1. Doelwit van die Projek

**Doelwit:** Speletjie-ontwikkelaars en beleggers help om 'n ingelige besluit te neem oor watter genre om in te belê, gebaseer op werklike Steam-data oor gewildheid, sentiment en suksesfaktore per genre.

**Vraag:** Watter genre werk op Steam?

**Hipotese:** Genre bepaal die basislyn-verwagtinge van spelers; suksesfaktore verskil tussen genres.

### Fase 2 se bydrae tot die doelwit

Fase 2 fokus op die verkryging en strukturering van die datastel:

1. **Aktiewe data-insameling** -- webskraping van Steam se openbare API.
2. **Data skoonmaak** -- probleme identifiseer en hanteer (empiries geregverdig).
3. **Voorbereiding** -- kenmerk-ingenieurswese, genre-etikettering, Engels-only filter.
4. **Dokumentasie** -- elke stap + redes; elke kolom gemotiveer met die navorsingsvraag.

**Uitkoms:** 'n skoon, Engelse datastel (`data/processed/reviews_clean.csv`) gereed vir Fase 3.

---
## 2. Beplanning (Klaswerk)

### Navorsingsvraag & Hipotese
**Vraag:** Watter genre werk op Steam?
**Hipotese:** Genre bepaal die basislyn-verwagtinge van spelers; suksesfaktore verskil tussen genres.

### Steekproef-ontwerp
33 speletjies oor 15 genres — elke genre het genoeg Engelse volume vir statistiese toetse.

### Skraapbesluite (na oorweging)

| Besluit | Redes |
|---|---|
| **Engels-only** | VADER en TF-IDF werk slegs op Engels; nie-Engelse resensies sou in Fase 4 ge-filter word. Doeltreffender skraping, skoner data. |
| **50 bladsye (5 000 resensies)** | Die nuutste 5 000 resensies dek ~1 jaar vir die meeste speletjies. Dit is recent genoeg om huidige gewildheid te meet, maar stabiel genoeg vir statistiek. Te min (<20 bladsye) is te veranderlik; te veel (>100) is te oud vir die hipotese. |
| **0.3 s vertraging** | Vriendelik teenoor die API; voldoende tyd tussen versoeke. |

### Skoonmaakplan
Empiries geregverdig in §5.1: minimum lengte, karakter-ratio, uitskieters-afkap.

### Etiese oorwegings
Slegs openbare data. Geen privaatgebruikersinligting. Geen manipulasie.

---
## 3. Opstelling

In [1]:
%matplotlib inline
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import os, time, inspect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import GAMES, GAME_IDS, RAW_DIR, PROCESSED_DIR, GENRES, GAME_GENRES, load_raw
from src import scrape, clean

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)

print('Modules ingevoer.  |  ', len(GAMES), 'speletjies, ', len(GENRES), 'genres')
for g, ids in GENRES.items():
    print(f'  {g:15s} ({len(ids):>2}): {", ".join(GAMES[a] for a in ids)}')

Modules ingevoer.  |   33 speletjies,  12 genres
  RPG             ( 8): Baldur's Gate 3, Cyberpunk 2077, Elden Ring, The Witcher 3, Skyrim SE, Dragon's Dogma 2, Fallout 4, Valheim
  Shooter         (11): Counter-Strike 2, PUBG: BATTLEGROUNDS, Helldivers 2, Call of Duty HQ, Overwatch 2, Apex Legends, Destiny 2, Rainbow Six Siege, Battlefield 2042, Battlefield 6, Team Fortress 2
  Hero_Shooter    ( 3): Overwatch 2, Marvel Rivals, Team Fortress 2
  Battle_Royale   ( 3): PUBG: BATTLEGROUNDS, Call of Duty HQ, Apex Legends
  Action          (17): Elden Ring, Cyberpunk 2077, No Man's Sky, Helldivers 2, Ghost of Tsushima, God of War, Red Dead Redemption 2, Devil May Cry 5, The Witcher 3, Skyrim SE, Dragon's Dogma 2, Fallout 4, Team Fortress 2, Rust, ARK: Survival Evolved, Valheim, The Forest
  Strategy        ( 6): Baldur's Gate 3, Stellaris, Sid Meier's Civilization VI, Total War: WARHAMMER III, Age of Empires IV, Rainbow Six Siege
  Adventure       (11): Baldur's Gate 3, No Man's Sky, The W

---
## 4. Aktiewe Data-Insameling: Steam API

Ons het die data **self geskraap** — geen klaargemaakte CSV afgelaai nie.

### Skraping-metode

| Parameter | Waarde | Regverdiging |
|---|---|---|
| **Taal** | `english` | VADER en TF-IDF werk slegs op Engels |
| **Bladsye** | 50 (5 000 resensies) | Die nuutste 5 000 dek ~1 jaar vir 31/33 speletjies — recent genoeg vir gewildheid, stabiel genoeg vir statistiek |
| **Aankoopspeletjie** | `purchase_type=all` | Alle resensies, nie net betaalde nie |
| **Tydreeks** | `day_range=9999` | Alle datums, maar paginering beperk naturaliter tot die nuutste |
| **Tempo** | 0.3 s | Vriendelik teenoor die API |
| **Deduplikasie** | `seen_ids` | Verhoed dat dieselfde resensie twee keer gestoor word |
| **Stop-reël** | 3 leë bladsye | Einde van die speletjie se resensies |

Die volle skraper-kode word hieronder gewys (src/scrape.py).

In [3]:
# Skraper: Bronze-kernfunksie — vertoon die kode wat die data insamel
print(inspect.getsource(scrape.scrape_reviews))

def scrape_reviews(app_id, max_pages=50, reviews_per_page=100):
    all_reviews = []
    cursor = '*'
    seen_ids = set()
    empty_page_count = 0

    for page in range(max_pages):
        params = {
            'json': 1,
            'language': 'english',
            'num_per_page': reviews_per_page,
            'purchase_type': 'all',
            'day_range': 9999,
            'cursor': cursor,
        }
        data = safe_request(f'{STEAM_REVIEWS_URL}/{app_id}', params)
        if not data or not data.get('success'):
            break

        reviews = data.get('reviews', [])
        if not reviews:
            break

        new_count = 0
        for r in reviews:
            rid = r.get('recommendationid', '')
            if rid in seen_ids:
                continue
            seen_ids.add(rid)
            new_count += 1
            all_reviews.append({
                'app_id': app_id,
                'game_name': GAMES.get(app_id, ''),
                'review_id': rid,
   

### 4.1 Lewendige demonstrasie

Skraap 2 bladsye vanaf die Steam API om te wys die tegniek werk.

In [ ]:
print('Skraap Elden Ring (2 bladsye)...')
try:
    demo = scrape.scrape_reviews(1245620, max_pages=2)
    print(f'{len(demo)} resensies ontvang')
    if demo:
        d = pd.DataFrame(demo)
        print(d[['game_name','language','voted_up','playtime_forever']].head(3).to_string(index=False))
        print('Voorbeeld:', repr(d['review_text'].iloc[0][:80]))
except Exception as e:
    print(f'Geen internet? {type(e).__name__}: {e}')

### 4.2 Die volle rou datastel

Die volle skraping is vroeër uitgevoer en gestoor in `data/raw/`. Ons laai dit in en filter na Engels.

In [ ]:
raw_path = f'{RAW_DIR}/reviews.csv'
df_raw_all = pd.read_csv(raw_path) if os.path.exists(raw_path) else None

if df_raw_all is not None:
    print(f'Rou data (alle tale): {len(df_raw_all):,} resensies, {df_raw_all["app_id"].nunique()} speletjies')
    print(f'Tale: {df_raw_all["language"].nunique()}\n')
    print('Taalverspreiding (top 10):')
    print(df_raw_all['language'].value_counts().head(10).to_string())

    # Filter na Engels
    df_raw = df_raw_all[df_raw_all['language'] == 'english'].copy()
    print(f'\nGefilter na Engels: {len(df_raw):,} resensies ({len(df_raw)/len(df_raw_all)*100:.1f}% van totaal)')
    print(f'Gemiddeld per speletjie: {len(df_raw)/df_raw["app_id"].nunique():.0f}')

    # Volledigheidstoets
    missing = [a for a in GAME_IDS if a not in set(df_raw['app_id'])]
    print(f'Speletjies met Engelse data: {df_raw["app_id"].nunique()}/{len(GAME_IDS)}')
    if missing: print(f'   Kort: {missing}')
else:
    print('Geen rou data. Hardloop: from src.scrape import scrape_all; scrape_all()')

df_details = load_raw('app_details.csv')
if df_details is not None:
    print(f'\napp_details: {len(df_details)} speletjies')

In [ ]:
print('Skraap huidige spelertellings (33 speletjies)...')
rows = []
try:
    for app_id in GAME_IDS:
        pc = scrape.fetch_player_count(app_id)
        if pc: rows.append(pc)
        time.sleep(0.3)
    df_players = pd.DataFrame(rows)
    df_players.to_csv(f'{RAW_DIR}/player_counts.csv', index=False)
    print(f'{len(df_players)} spelertellings gestoor')
    print(df_players.sort_values('player_count', ascending=False).head(5).to_string(index=False))
except Exception as e:
    print(f'Geen internet: {e}')

### 4.3 Volume: 50 bladsye = ~1 jaar

Waarom **50 bladsye** (5 000 resensies) en nie meer of minder nie?

Die Steam API gee resensies in omgekeerde kronologiese volgorde (nuutste eerste). Die nuutste 5 000 resensies dek **~1 jaar vir die meeste speletjies**. Dit is die regte skaal vir ons hipotese (“Watter genre is **huidiglik** gewild?”):

In [ ]:
# Toon: hoekom 50 bladsye (5 000 resensies) die regte skaal is
print('=== DATUMBEREIK OP VERSKILLENDE HOOFGROOTTETES (alle 33 speletjies) ===\n')
print(f'{"Hoofgrootte":>12s}  {"Gem. dae":>10s}  {"Med. dae":>10s}  {"Min dae":>10s}  {"Max dae":>10s}')
print('-' * 70)
for n in [1000, 2000, 3000, 5000, 10000]:
    spans = []
    for g in df_raw['game_name'].unique():
        sub = df_raw[df_raw['game_name']==g].sort_values('timestamp_created', ascending=False)
        s = sub.head(n)
        if len(s) >= 100:
            spans.append((s['timestamp_created'].max() - s['timestamp_created'].min()) / 86400)
    if spans:
        print(f'{n:>12,}  {np.mean(spans):>10.0f}  {np.median(spans):>10.0f}  {min(spans):>10.0f}  {max(spans):>10.0f}')

print('\n=== PER-SPELTTJE: Engelse resensies + tydperk van nuutste 5 000 ===\n')
print(f'{"Speletjie":30s}  {"Engels":>7s}  {"5k dae":>6s}  Tydperk')
print('-' * 80)
game_eng = df_raw.groupby('game_name').agg(
    eng_count=('review_id','count'),
    min_ts=('timestamp_created','min'),
    max_ts=('timestamp_created','max')
).sort_values('eng_count', ascending=False)
from datetime import datetime as dt
for game, row in game_eng.iterrows():
    n5k = df_raw[df_raw['game_name']==game].sort_values('timestamp_created', ascending=False).head(5000)
    days = (n5k['timestamp_created'].max() - n5k['timestamp_created'].min()) / 86400 if len(n5k) > 1 else 0
    d1 = dt.fromtimestamp(n5k['timestamp_created'].min())
    d2 = dt.fromtimestamp(n5k['timestamp_created'].max())
    flag = ''
    print(f'{game:30s}  {row["eng_count"]:>7,}  {days:>5.0f}  {d1:%Y-%m-%d} tot {d2:%Y-%m-%d}{flag}')

print('\nGevolgtrekking: 50 bladsye (5 000 resensies) dek ~1 jaar vir die meeste speletjies.')
print('Wanneer die skraper na Engels-only oorskakel, sal alle speletjies tot 5 000 Engelse resensies kry.')

---
## 5. Probleme in die Rou Data

Die skoonmaakparameters is **empiries** bepaal — nie arbitrêr nie.

In [ ]:
print('=== Probleme in die Engelse rou data ===\n')

n_dup = df_raw['review_id'].duplicated().sum()
print(f'1. Duplikaat review_id:          {n_dup}')

empty_mask = df_raw['review_text'].isna() | (df_raw['review_text'].str.strip() == '')
print(f'2. Leë/ontbrekende teks:         {empty_mask.sum()}')

short_mask = (~empty_mask) & (df_raw['review_text'].str.len() < 10)
print(f'3. Korter as 10 karakters:        {short_mask.sum()}')

def ratio(s):
    s = str(s)
    return sum(c.isalpha() or c.isspace() for c in s) / max(len(s), 1)
r = df_raw['review_text'].fillna('').apply(ratio)
print(f'4. Nonsens (>50% nie-alfabeties): {(r <= 0.5).sum()}')

print(f'5. Speeltyd: maks {df_raw["playtime_forever"].max():,.0f} min | 99ste pct: {df_raw["playtime_forever"].quantile(0.99):,.0f} min')

print(f'6. Ongeldige tydstempels:         {pd.to_numeric(df_raw["timestamp_created"], errors="coerce").isna().sum()}')

print('\nVoorbeelde kort resensies:')
for t in df_raw[short_mask]['review_text'].dropna().value_counts().head(6).index:
    print(f'  "{t}"')

### 5.1 Empiriese parameter-regverdiging

Al drie skoonmaak-parameters is met getalle geregverdig — nie op gevoel gekies nie.

In [ ]:
print('=== PARAMETER 1: Minimum resensielengte ===')
print('Waarom 10 karakters en nie 5 of 15?\n')
lengths = df_raw['review_text'].fillna('').str.len()
print(f'{"Drempel":>12s}  {"Verwyder":>10s}  {"%":>7s}  Wat kry ons weg?')
print('-' * 65)
for n in [5, 8, 10, 12, 15]:
    cnt = (lengths < n).sum()
    pct = cnt / len(df_raw) * 100
    examples = df_raw[lengths < n]['review_text'].dropna().value_counts().head(2).index.tolist()
    ex_str = ', '.join(repr(e) for e in examples[:2])
    print(f'  < {n:>2} karakters: {cnt:>6}  ({pct:.2f}%)  {ex_str}')

print('\nGevolgtrekking: <10 is die sweet spot — vang spam ("good", "nice", ".") sonder om')
print('geldige kort resensies (bv. "Great game, love it") te verloor.')

print('\n' + '='*65)
print('=== PARAMETER 2: Karakter-ratio drempel ===')
print('Waarom >50% alfabeties en nie 30% of 70%?\n')
print(f'{"Drempel":>12s}  {"Verwyder":>10s}  {"%":>7s}  Skuiwe vs vorige')
print('-' * 55)
prev = 0
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cnt = (r <= t).sum()
    delta = cnt - prev
    print(f'  >{t:.1f} alpha:    {cnt:>6}  ({cnt/len(df_raw)*100:.2f}%)  +{delta} vanaf vorige')
    prev = cnt

print('\nGevolgtrekking: 0.5 vang emoji-spam en nonsens sonder om hemelings')
print('te verloor. Die spronge tussen drempels is klein — 0.5 is veilig.')

---
## 6. Skoonmaak en Voorbereiding (`src/clean.py`)

| # | Stap | Wat | Waarom |
|---|---|---|---|
| 1 | Deduplikasie | `review_id` dubbels verwyder | Dubbeltelling |
| 2 | Leë teks | NaN/leë `review_text` weg | Geen inligting |
| 3 | Min lengte | < 10 karakters verwyder | Spam (empiries geregverdig) |
| 4 | Karakter-ratio | >50% alfabeties/spasies | Emoji/nonsens-spam (empiries) |
| 5 | Tydstempels | Unix → `review_date` etc. | Datums vir konteks |
| 6 | Speeltyd | Uitskieters by 99ste pct afgekap | Behou resensie, neutraliseer ekstreme |
| 7 | Tipes | `voted_up` → bool, playtime → float | Korrekte datatipes |
| 8 | Teks-skoonmaak | URL's, spesiale karakters weg | Skoon teks vir NLP |
| 9 | Kenmerke | `word_count`, `review_length` | Nuwe kenmerke |
| 10 | Genre-one-hot | 15 × `genre_{GENRE}` | Genre-vlak analise |

**Belangrike besluite:** Uitskieters word **afgekap** (nie verwyder nie); ontbrekende speeltyd → 0.

In [ ]:
print(inspect.getsource(clean.clean_reviews))

In [ ]:
t0 = time.time()
df_clean = clean.clean_reviews(df_raw)  # df_raw is reeds Engels-only (uit §4.2)
dt = time.time() - t0

print(f'\nSkoonmaak voltooi in {dt:.1f}s')
print(f'Voor: {len(df_raw):,} Engelse resensies')
print(f'Na:   {len(df_clean):,} skoon resensies ({(len(df_raw)-len(df_clean))/len(df_raw)*100:.2f}% verwyder)')
print(f'Kolomme: {df_clean.shape[1]}')

# Trechter-visualisering
_em = df_raw['review_text'].isna() | (df_raw['review_text'].str.strip() == '')
_sm = (~_em) & (df_raw['review_text'].str.len() < 10)
funnel = [len(df_raw), int((~_em).sum()), int((~_em & ~_sm).sum()), len(df_clean)]
labels = ['Rou (Engels)', 'Nie-leeg', 'Lengte>=10', 'Karakter-ratio']
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(labels, funnel, color=['#1b2838','#2a475e','#66c0f4','#a4d007'], width=0.6)
ax.set_ylabel('Aantal resensies')
ax.set_title(f'Skoonmaak-trechter: {funnel[0]:,} -> {funnel[-1]:,}', fontweight='bold')
ax.spines[['top','right']].set_visible(False)
for b, v in zip(bars, funnel):
    ax.text(b.get_x()+b.get_width()/2, v+300, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

df_clean[['game_name','review_date','playtime_forever','voted_up','word_count','review_text_clean']].head(3)

### 6.1 Genre-one-hot

Elke resensie kry one-hot-aanwysers vir al sy speletjie se genres (uit `GAME_GENRES`).
So kan ons per-genre vergelyk sonder om speletjies met meerdere genres te dubbeltel.

In [ ]:
genre_cols = [c for c in df_clean.columns if c.startswith('genre_')]
print(f'Genre-aanwysers ({len(genre_cols)}): {genre_cols}\n')
print('Voorbeeld GAME_GENRES-kartering:')
for a in [1245620, 730, 252490]:
    print(f'  {GAMES[a]:25s}: {GAME_GENRES.get(a)}')

print('\nResensies per genre (speletjies kan in meerdere genres wees):')
for g, ids in GENRES.items():
    print(f'  {g:15s}: {df_clean[df_clean["app_id"].isin(ids)].shape[0]:>7,}')

### 6.2 Hoekom one-hot eerder as Genre-pare?

**One-hot encoding** gee elke genre sy eie kolom (0 of 1). "n Speletjie kan in verskeie genres val:

| Speletjie | RPG | Shooter | Action | Survival |
|---|---|---|---|---|
| Elden Ring | **1** | 0 | **1** | 0 |
| CS2 | 0 | **1** | 0 | 0 |
| Valheim | **1** | 0 | **1** | **1** |

**Waarom one-hot beter is as pare:**

1. **Onafhanklike effekte:** "n Regressiemodel kan meet: "Hou ander dinge gelyk, gee RPG 'n boost aan positiewe resensies?" Pare kan nie dit doen nie.
2. **Geen dooie pare:** Met 15 genres is daar 66 moontlike pare. Die meeste het 0-2 speletjies. One-hot benut al 33 speletjies.
3. **Volledige informasie:** Elden Ring se RPG + Action status word nie verlore nie.

**Waarom pare nog steeds nuttig is:**

One-hot meet elke genre se *individuele* bydrae. Maar party genres kom altyd saam voor (bv. RPG + Action). Genre-pare wys watter **kombinasies** werk — "n vraag wat one-hot alleen nie kan beantwoord nie.

Ons doen **albei**: one-hot vir regressie, pare vir vergelykende analise van die sterkste kombinasies.

In [ ]:
from itertools import combinations

print('=== Genre-pare met >= 2 speletjies ===\n')
pairs = []
for g1, g2 in combinations(GENRES.keys(), 2):
    overlap = set(GENRES[g1]) & set(GENRES[g2])
    if len(overlap) >= 2:
        names = [GAMES[a] for a in overlap]
        pairs.append((len(overlap), g1, g2, names))

pairs.sort(key=lambda x: -x[0])
print(f'{"Grootte":>7s}  Paar                        Speletjies')
print('-' * 80)
for size, g1, g2, names in pairs:
    label = f'{g1} + {g2}'
    print(f'{size:>7d}  {label:30s}  {", ".join(names)}')

print(f'\nTotaal pare met >= 2 speletjies: {len(pairs)}')
print('Analise in Fase 4: vergelyk positiewe %, sentiment, speeltyd per sterkste paar.')

### 6.3 Stoor die skoon datastel

VADER-kolomme van die vorige loping word bewaar om herberekening (~50 min) te vermy. VADER self is Fase 4.

In [ ]:
out_path = f'{PROCESSED_DIR}/reviews_clean.csv'

# Bewaar VADER van vorige loping
vader_cols = [c for c in df_clean.columns if c.startswith('vader_')]
if not vader_cols and os.path.exists(out_path):
    prev_cols = pd.read_csv(out_path, nrows=0).columns
    vader_cols = [c for c in prev_cols if c.startswith('vader_')]
    if vader_cols:
        prev = pd.read_csv(out_path, usecols=['review_id'] + vader_cols)
        df_clean = df_clean.merge(prev, on='review_id', how='left')
        print(f'VADER bewaar: {vader_cols}')

df_clean.to_csv(out_path, index=False)
print(f'Gestoor: {out_path}  |  {len(df_clean):,} rye  |  {df_clean.shape[1]} kolomme')

---
## 7. Finale Datastel Struktuur

In [ ]:
print('=== Kolomme en datatipes ===')
print(df_clean.dtypes.to_string())

key_cols = ['review_id','review_text','timestamp_created','playtime_forever','voted_up']
missing = df_clean[key_cols].isna().sum()
print('\n=== Ontbrekende waardes (kernkolomme) ===')
print(missing.to_string() if missing.any() else 'Geen ontbrekende waardes.')

print('\n=== Per-speletjie-oorsig ===')
pg = df_clean.groupby('game_name').agg(
    resensies=('review_id','count'),
    positief_pct=('voted_up','mean'),
    gem_speeltyd=('playtime_forever','mean'),
    gem_woorde=('word_count','mean')
).sort_values('resensies', ascending=False)
print(pg.to_string())

from datetime import datetime as dt
print(f'\nDatumbereik: {dt.fromtimestamp(df_raw["timestamp_created"].min()):%Y-%m-%d} tot {dt.fromtimestamp(df_raw["timestamp_created"].max()):%Y-%m-%d}')

### 7.1 Kolom-motivasie: elke kolom geregverdig met die navorsingsvraag

Die navorsingsvraag is **“Watter genre werk?”** — elke kolom moet bydra tot hierdie vraag:

| Kolom(mes) | Rede |
|---|---|
| `app_id`, `game_name` | Identifikasie: watter speletjie? |
| `genre_*` (15 kolomme) | **Kern van die projek** — genre-aanwysers vir per-genre analise |
| `voted_up` | **Die teiken**: is die resensie positief? (sukses-maatstaf) |
| `language` | Filtro vir Engels-only (VADER/TF-IDF-vereiste) — **verwyder** na filter (kolom is nou konstant) |
| `playtime_forever` | Speler-betrokkenheid — voorspel sentiment; sterk kenmerk per hipotese |
| `playtime_at_review` | **Verwyder** — nie gebruik in enige analise nie |
| `review_text`, `review_text_clean` | Rou/skoon teks vir NLP (Fase 4) |
| `review_length`, `word_count` | Resensielengte — correleer met sentiment; onderskei kritiek vs lof |
| `review_date`, `review_year/month/day_of_week` | Temporele konteks vir datumbereik en seisoenale redes |
| `vader_*` (5 kolomme) | Sentiment-analise (Fase 4, hier bewaar van vorige berekening) |
| `votes_up`, `votes_funny` | Gemeenskapsreaksie — meet watter resensies opvallend is |
| `steam_purchase` | Konteksfaktor: betaalde vs gratis-aankoop kan tevredenheid beïnvloed (`is_steam_purchase` was duplikaat — **verwyder**) |
| `written_during_early_access` | Konteksfaktor: EA-speletjies het ander verwagtinge (`has_early_access` was duplikaat — **verwyder**) |
| `weighted_vote_score` | Steam se geweegde tellings — nuttig om sigbaarheid van resensies te meet |
| `num_games_owned`, `num_reviews` | Speler-ervaring: veteranes vs nuwelings mag verskil in kritiek |
| `received_for_free` | Konteksfaktor: gratis-spesifieke verwagtinge |
| `timestamp_created` | Tydstempels vir afgeleide datums + potensiële seisoenale analise (`timestamp_updated` is **verwyder** — nie gebruik nie) |

**Verwyder:** `language` (konstant na filter), `timestamp_updated`, `playtime_at_review` (nooit gebruik nie), `has_early_access`/`is_steam_purchase` (duplikate van bestaande kolomme).
**Aantekening:** Vir die volgende fase kan kolomme soos `received_for_free` en `num_games_owned`,
indien hulle lae veranderlikheid toon, oorweeg word vir verwydering om model-kompleksiteit te verminder.

---
## 8. Datakwaliteit-opsomming

In [ ]:
print('=== Datakwaliteit-opsomming (Engels-only) ===')
print(f'Rou (alle tale):     {len(df_raw_all):>7,}')
print(f'Rou (Engels):        {len(df_raw):>7,}')
print(f'Skoon resensies:     {len(df_clean):>7,}')
print(f'Verwyder uit Engels: {len(df_raw)-len(df_clean):>7,} ({(len(df_raw)-len(df_clean))/len(df_raw)*100:.1f}%)')
print(f'Speletjies:          {df_clean["app_id"].nunique():>7}/{len(GAME_IDS)}')
from datetime import datetime as dt
print(f'Datumreeks:          {dt.fromtimestamp(df_raw["timestamp_created"].min()):%Y-%m-%d} tot {dt.fromtimestamp(df_raw["timestamp_created"].max()):%Y-%m-%d}')
print(f'Positief:            {df_clean["voted_up"].mean()*100:.1f}%')
print(f'Gem. woordelengte:   {df_clean["word_count"].mean():.0f}')
print(f'Genre-aanwysers:     {len([c for c in df_clean.columns if c.startswith("genre_")])}')
print(f'VADER-kolomme:       {len([c for c in df_clean.columns if c.startswith("vader_")])}')
print('\nStatus: GEREED vir Fase 3 — Engels-only, gestruktureer, geregverdig.')

---
## 9. Plan vir Fase 3: Verkennende Data Analise (EDA)

**Hierdie fase word slegs beplan — nie uitgevoer nie.**

1. **Basiese statistieke:** resensies, positief-%, speeltyd per speletjie en per genre.
2. **Formele toetse:** t-toetse (resensielengte/speeltyd volgens stemming), ANOVA (verskille tussen genres).
3. **Patrone:** volume, sentiment-verspreiding, per-genre-verskille.
4. **Aanvanklike plan Fase 4:** VADER, TF-IDF, netwerkgrafieke, regressie, dashboard.

Die bevindinge van Fase 3 rig die finale Fase 4-benadering.

---
## 10. Bronnelys

1. **Steam API** — https://steamcommunity.com/dev
2. **Steam Web API (appreviews)** — https://partner.steamgames.com/doc/store/getreviews
3. **pandas** — McKinney, W. (2010). Data Structures for Statistical Computing in Python. *9th Python in Science Conference*.
4. **requests** — https://requests.readthedocs.io
5. **VADER-Sentiment** — Hutto, C.J. & Gilbert, E.E. (2014). VADER: A Parsimonious Rule-based Model for Sentiment Analysis of Social Media Text. *ICWSM-14*.

---
*Fase 2 — Portefeulje Projek. Alle data is openbare Steam-data. Naatlik aangepas na professor se terugvoer.*